In [1]:
# ============================================================
# Per-dimension proxy penalization escape sweep
# AlpacaEval + cached ArmoRM scores
# ============================================================

# Cell 1 — Install packages
!pip install -q -U scipy textstat tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 33.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.1/177.1 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 676.6/676.6 kB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 41.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydata-profiling 4.18.4 requires scipy<1.17,>=1.8, but you have scipy 1.18.0 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.


In [2]:
# Cell 2 — Imports and config
import json, re, math, warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import textstat
from tqdm.auto import tqdm
from scipy.stats import spearmanr

warnings.filterwarnings("ignore")

POOL_PATH = Path("/kaggle/input/datasets/pradeep01223/alpacaeval-500x256-llama8b/pool_candidates_merged_300.jsonl")
SCORE_PATH = Path("/kaggle/input/notebooks/rjjack0112/llama-alpaceval-armo-score/armorm_cache_128k.csv")

OUT_DIR = Path("/kaggle/working/exp1_3_per_dimension_escape")
OUT_DIR.mkdir(parents=True, exist_ok=True)

N_LIST = [1, 4, 16, 64, 256]
PENALTY_WEIGHT = 0.35
ALPHA = 0.05
MIN_RHO = 0.5
STATE_MAP = {"POSITIVE": "p", "NEGATIVE": "n"}

KEYWORD_SET = {
    "effective","important","key","great","best","helpful","excellent","significant",
    "valuable","essential","improve","benefit","result","optimize","performance",
    "crucial","critical","powerful","robust","strong","useful","efficient","successful",
    "positive","major","notable","remarkable","outstanding","superior","ideal","optimal",
    "productive","impactful","meaningful","achieve","enhance","boost","increase",
    "maximize","support","enable","ensure","provide","offer"
}
HEDGING_WORDS = {
    "may","might","could","can","possibly","perhaps","arguably","sometimes","often",
    "typically","generally","likely","unlikely","suggests","suggest","appears","appear",
    "seems","seem","roughly","approximately","about"
}
CERTAINTY_WORDS = {
    "definitely","certainly","clearly","obviously","always","never","must","proves",
    "prove","undeniably","absolutely","guaranteed","guarantee"
}
STOPWORDS = {
    "the","a","an","and","or","but","if","then","than","that","this","these","those",
    "to","of","in","on","for","with","as","by","is","are","was","were","be","been",
    "being","it","its","at","from","into","about","over","after","before","between",
    "through","during","under","above","below","up","down","out","off","again","further",
    "once","here","there","when","where","why","how","all","any","both","each","few",
    "more","most","other","some","such","no","nor","not","only","own","same","so",
    "too","very","s","t","can","will","just","don","should","now","i","you","he","she",
    "we","they","them","their","our","your","my","me","his","her"
}

In [3]:
# Cell 3 — Output paths
CANDIDATE_METRICS_CSV = OUT_DIR / "exp1_3_candidate_metrics.csv"
SELECTIONS_CSV = OUT_DIR / "exp1_3_selections.csv"
SUMMARY_CSV = OUT_DIR / "exp1_3_summary.csv"
ESCAPE_MATRIX_CSV = OUT_DIR / "exp1_3_escape_matrix.csv"
REPORT_TXT = OUT_DIR / "exp1_3_report.txt"

In [4]:
# Cell 4 — Helper functions (per-candidate metrics)
def safe_get_prompt_id(rec):
    if "prompt_id" in rec: return int(rec["prompt_id"])
    if "promptid" in rec: return int(rec["promptid"])
    raise KeyError("No prompt_id/promptid field found.")

def safe_get_instruction(rec):
    for k in ["instruction","prompt","question","input"]:
        if k in rec: return rec[k]
    return ""

def tokenize_words(text): return re.findall(r"\b[a-zA-Z']+\b", str(text).lower())

def response_length(text): return len(str(text).split())

def repetition_score(text):
    toks = str(text).lower().split()
    if len(toks) < 2: return 0.0
    bigrams = list(zip(toks, toks[1:]))
    if not bigrams: return 0.0
    return 1.0 - (len(set(bigrams)) / len(bigrams))

def keyword_density(text):
    words = tokenize_words(text)
    return sum(1 for w in words if w in KEYWORD_SET) / max(1, len(words))

def hedging_density(text):
    words = tokenize_words(text)
    return sum(1 for w in words if w in HEDGING_WORDS) / max(1, len(words))

def certainty_density(text):
    words = tokenize_words(text)
    return sum(1 for w in words if w in CERTAINTY_WORDS) / max(1, len(words))

def list_density(text):
    lines = str(text).splitlines()
    bullet_like = sum(1 for ln in lines if re.match(r"^\s*[-*•]\s+", ln) or re.match(r"^\s*\d+[.)]\s+", ln))
    return bullet_like / max(1, len(lines))

def formatting_complexity(text):
    txt = str(text); lines = txt.splitlines()
    bullets = sum(1 for ln in lines if re.match(r"^\s*[-*•]\s+", ln) or re.match(r"^\s*\d+[.)]\s+", ln))
    markdown_marks = txt.count("**") + txt.count("##") + txt.count("```") + txt.count("|")
    colon_lines = sum(1 for ln in lines if ":" in ln)
    return (bullets + markdown_marks + 0.25 * colon_lines) / max(1, len(lines))

def flesch_inverse(text):
    try: f = float(textstat.flesch_reading_ease(str(text)))
    except Exception: f = 50.0
    return -f

def type_token_ratio(text):
    words = tokenize_words(text)
    if not words: return 0.0
    return len(set(words)) / len(words)

def lexical_rarity(text):
    words = [w for w in tokenize_words(text) if w not in STOPWORDS]
    if not words: return 0.0
    counts = Counter(words)
    return sum(1 for w,c in counts.items() if c==1) / max(1, len(counts))

DIM_FUNCS = {
    "length": response_length,
    "repetition": repetition_score,
    "keyword_density": keyword_density,
    "hedging_density": hedging_density,
    "certainty_density": certainty_density,
    "list_density": list_density,
    "formatting_complexity": formatting_complexity,
    "flesch_inverse": flesch_inverse,
    "type_token_ratio": type_token_ratio,
    "lexical_rarity": lexical_rarity,
}
SECONDARY_DIMS = list(DIM_FUNCS.keys())

def zscore(s):
    s = pd.Series(s, dtype=float)
    std = s.std(ddof=0)
    if std == 0 or np.isnan(std): return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - s.mean()) / std

def positive_part(x): return np.maximum(x, 0.0)

In [5]:
# Cell 5 — Load pool + score cache
assert POOL_PATH.exists(), f"Missing pool file: {POOL_PATH}"
assert SCORE_PATH.exists(), f"Missing score file: {SCORE_PATH}"

pool_records = []
with open(POOL_PATH, "r", encoding="utf-8") as f:
    for line in f:
        rec = json.loads(line)
        pool_records.append({
            "promptid": safe_get_prompt_id(rec),
            "instruction": safe_get_instruction(rec),
            "candidates": rec["candidates"],
            "n_candidates": len(rec["candidates"])
        })

score_df = pd.read_csv(SCORE_PATH)
score_df.columns = [c.strip() for c in score_df.columns]
score_df["promptid"] = score_df["promptid"].astype(int)
score_df["candidx"] = score_df["candidx"].astype(int)
score_df["armorm_raw"] = score_df["armorm_raw"].astype(float)
score_map = {(int(r.promptid), int(r.candidx)): float(r.armorm_raw) for r in score_df.itertuples(index=False)}

print(f"Prompts loaded: {len(pool_records)} | Score rows: {len(score_df)}")

Prompts loaded: 500 | Score rows: 128000


In [6]:
# Cell 6 — Compute per-candidate metrics for ALL candidates (cached once)
rows = []
for rec in tqdm(pool_records, desc="Computing candidate metrics"):
    pid = rec["promptid"]
    for idx, cand in enumerate(rec["candidates"]):
        if (pid, idx) not in score_map:
            continue
        row = {"promptid": pid, "candidx": idx, "armorm_raw": score_map[(pid, idx)]}
        for dim, fn in DIM_FUNCS.items():
            row[dim] = fn(cand)
        rows.append(row)

metrics_df = pd.DataFrame(rows)
metrics_df.to_csv(CANDIDATE_METRICS_CSV, index=False)
print(f"Saved candidate metrics: {CANDIDATE_METRICS_CSV} | shape={metrics_df.shape}")

Computing candidate metrics:   0%|          | 0/500 [00:00<?, ?it/s]

Saved candidate metrics: /kaggle/working/exp1_3_per_dimension_escape/exp1_3_candidate_metrics.csv | shape=(128000, 13)


In [7]:
# Cell 7 — Per-prompt z-scored dimensions (for penalty construction)
zscored = metrics_df.copy()
for dim in SECONDARY_DIMS:
    zscored[f"{dim}_z"] = zscored.groupby("promptid")[dim].transform(zscore)
zscored.to_csv(OUT_DIR / "exp1_3_candidate_metrics_z.csv", index=False)

# Cell 8 — BoN selection sweep: one condition per penalized dimension + control
def select_bon(df_prompt, score_col, N):
    sub = df_prompt[df_prompt["candidx"] < N]
    if sub.empty: return None
    return sub.loc[sub[score_col].idxmax()]

all_rows = []
conditions = ["control"] + [f"penalize_{d}" for d in SECONDARY_DIMS]

grouped = zscored.groupby("promptid")

for pid, dfp in tqdm(grouped, desc="BoN sweep per condition"):
    dfp = dfp.copy()
    dfp["score_control"] = dfp["armorm_raw"]
    for dim in SECONDARY_DIMS:
        penalty = PENALTY_WEIGHT * positive_part(dfp[f"{dim}_z"])
        dfp[f"score_penalize_{dim}"] = dfp["armorm_raw"] - penalty

    for cond in conditions:
        score_col = "score_control" if cond == "control" else f"score_{cond}"
        for N in N_LIST:
            sel = select_bon(dfp, score_col, N)
            if sel is None: continue
            row = {"promptid": pid, "condition": cond, "N": N, "candidx": int(sel["candidx"]),
                   "armorm_raw": float(sel["armorm_raw"]), "selected_score": float(sel[score_col])}
            for dim in SECONDARY_DIMS:
                row[dim] = float(sel[dim])
            all_rows.append(row)

sel_df = pd.DataFrame(all_rows)
sel_df.to_csv(SELECTIONS_CSV, index=False)
print(f"Saved selections: {SELECTIONS_CSV} | shape={sel_df.shape}")

BoN sweep per condition:   0%|          | 0/500 [00:00<?, ?it/s]

Saved selections: /kaggle/working/exp1_3_per_dimension_escape/exp1_3_selections.csv | shape=(27500, 16)


In [8]:
# Cell 9 — Aggregate summary by condition and N
agg_dict = {"prompts": ("promptid", "nunique"),
            "armorm_raw_mean": ("armorm_raw", "mean"),
            "selected_score_mean": ("selected_score", "mean")}
for dim in SECONDARY_DIMS:
    agg_dict[f"{dim}_mean"] = (dim, "mean")

summary = sel_df.groupby(["condition", "N"]).agg(**agg_dict).reset_index()
summary.to_csv(SUMMARY_CSV, index=False)
print(f"Saved summary: {SUMMARY_CSV}")
display(summary)

Saved summary: /kaggle/working/exp1_3_per_dimension_escape/exp1_3_summary.csv


,condition,N,prompts,armorm_raw_mean,selected_score_mean,length_mean,repetition_mean,keyword_density_mean,hedging_density_mean,certainty_density_mean,list_density_mean,formatting_complexity_mean,flesch_inverse_mean,type_token_ratio_mean,lexical_rarity_mean
0,control,1,500,0.046552,0.046552,185.018,0.036286,0.008316,0.012354,0.001400,0.161508,0.583412,-37.712331,0.721611,0.854695
1,control,4,500,0.079568,0.079568,181.770,0.047098,0.008033,0.013269,0.000930,0.240125,0.779452,-41.525849,0.673385,0.825833
2,control,16,500,0.096433,0.096433,178.202,0.049141,0.008404,0.013498,0.000816,0.256495,0.859237,-42.760765,0.671965,0.830498
3,control,64,500,0.106938,0.106938,172.688,0.048703,0.008604,0.013004,0.000970,0.285076,0.891120,-43.242395,0.672683,0.827532
4,control,256,500,0.114835,0.114835,165.304,0.048207,0.008081,0.012452,0.001151,0.280411,0.863390,-42.326672,0.678413,0.826124
5,penalize_certainty_density,1,500,0.046552,-0.077357,185.018,0.036286,0.008316,0.012354,0.001400,0.161508,0.583412,-37.712331,0.721611,0.854695
6,penalize_certainty_density,4,500,0.075740,0.075392,182.006,0.047056,0.008171,0.013217,0.000123,0.229684,0.753657,-41.282357,0.674772,0.826670
7,penalize_certainty_density,16,500,0.094867,0.094717,178.380,0.049953,0.008406,0.013823,0.000104,0.253868,0.847646,-42.472851,0.671541,0.829027
8,penalize_certainty_density,64,500,0.105701,0.105687,172.830,0.049539,0.008542,0.012709,0.000067,0.280606,0.878041,-42.798163,0.672098,0.827028
9,penalize_certainty_density,256,500,0.113887,0.113887,164.580,0.048479,0.008108,0.012536,0.000092,0.279889,0.878132,-43.070876,0.677218,0.823336


In [9]:
# Cell 10 — Escape matrix: for each penalized dimension, check every OTHER dimension for escape
logN = [math.log2(n) for n in N_LIST]
escape_rows = []

for pen_dim in SECONDARY_DIMS:
    cond = f"penalize_{pen_dim}"
    block = summary[summary["condition"] == cond].sort_values("N")

    row_n1 = block.loc[block["N"] == 1]
    row_n256 = block.loc[block["N"] == 256]

    if row_n1.empty or row_n256.empty:
        continue

    pen_dim_n1 = float(row_n1[f"{pen_dim}_mean"].iloc[0])
    pen_dim_n256 = float(row_n256[f"{pen_dim}_mean"].iloc[0])
    score_n1 = float(row_n1["selected_score_mean"].iloc[0])
    score_n256 = float(row_n256["selected_score_mean"].iloc[0])

    penalized_dim_suppressed = pen_dim_n256 < pen_dim_n1
    score_recovered = score_n256 > score_n1

    for other_dim in SECONDARY_DIMS:
        if other_dim == pen_dim:
            continue

        vals = block[f"{other_dim}_mean"].tolist()
        rho, p = spearmanr(logN, vals)
        delta = vals[-1] - vals[0]

        escape_positive = (
            penalized_dim_suppressed and
            delta > 0 and
            rho >= MIN_RHO and
            p < ALPHA
        )

        escape_rows.append({
            "penalized_dimension": pen_dim,
            "escaped_to_dimension": other_dim,
            "penalized_dim_N1": pen_dim_n1,
            "penalized_dim_N256": pen_dim_n256,
            "penalized_dim_suppressed": penalized_dim_suppressed,
            "score_N1": score_n1,
            "score_N256": score_n256,
            "score_recovered": score_recovered,
            "other_dim_N1": vals[0],
            "other_dim_N256": vals[-1],
            "other_dim_delta": delta,
            "other_dim_spearman_rho": rho,
            "other_dim_p_value": p,
            "state": STATE_MAP["POSITIVE"] if escape_positive else STATE_MAP["NEGATIVE"]
        })

escape_df = pd.DataFrame(escape_rows)
escape_df = escape_df.sort_values(
    ["penalized_dimension", "state", "other_dim_delta"],
    ascending=[True, True, False]
)
escape_df.to_csv(ESCAPE_MATRIX_CSV, index=False)
print(f"Saved escape matrix: {ESCAPE_MATRIX_CSV}")
display(escape_df)

Saved escape matrix: /kaggle/working/exp1_3_per_dimension_escape/exp1_3_escape_matrix.csv


,penalized_dimension,escaped_to_dimension,penalized_dim_N1,penalized_dim_N256,penalized_dim_suppressed,score_N1,score_N256,score_recovered,other_dim_N1,other_dim_N256,other_dim_delta,other_dim_spearman_rho,other_dim_p_value,state
37,certainty_density,repetition,0.001400,0.000092,True,-0.077357,0.113887,True,0.036286,0.048479,0.012193,0.6,2.847570e-01,n
39,certainty_density,hedging_density,0.001400,0.000092,True,-0.077357,0.113887,True,0.012354,0.012536,0.000183,0.1,8.728886e-01,n
38,certainty_density,keyword_density,0.001400,0.000092,True,-0.077357,0.113887,True,0.008316,0.008108,-0.000208,-0.1,8.728886e-01,n
44,certainty_density,lexical_rarity,0.001400,0.000092,True,-0.077357,0.113887,True,0.854695,0.823336,-0.031359,-0.7,1.881204e-01,n
43,certainty_density,type_token_ratio,0.001400,0.000092,True,-0.077357,0.113887,True,0.721611,0.677218,-0.044393,-0.3,6.238377e-01,n
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,type_token_ratio,hedging_density,0.721611,0.649234,True,-0.101525,0.112621,True,0.012354,0.011952,-0.000402,-0.3,6.238377e-01,n
80,type_token_ratio,lexical_rarity,0.721611,0.649234,True,-0.101525,0.112621,True,0.854695,0.808006,-0.046690,-0.7,1.881204e-01,n
79,type_token_ratio,flesch_inverse,0.721611,0.649234,True,-0.101525,0.112621,True,-37.712331,-42.472962,-4.760631,-0.7,1.881204e-01,n
72,type_token_ratio,length,0.721611,0.649234,True,-0.101525,0.112621,True,185.018000,174.396000,-10.622000,-1.0,1.404265e-24,n


In [10]:
# Cell 11 — Per-dimension overall verdict
verdict_rows = []

for pen_dim in SECONDARY_DIMS:
    block = escape_df[escape_df["penalized_dimension"] == pen_dim]
    if block.empty:
        continue

    positive_escapes = block[block["state"] == STATE_MAP["POSITIVE"]]
    any_escape = len(positive_escapes) > 0
    top_escape = None

    if any_escape:
        top = positive_escapes.sort_values(
            ["other_dim_delta", "other_dim_spearman_rho"],
            ascending=[False, False]
        ).iloc[0]
        top_escape = top["escaped_to_dimension"]

    verdict_rows.append({
        "penalized_dimension": pen_dim,
        "any_escape_detected": any_escape,
        "n_escape_targets": int(len(positive_escapes)),
        "top_escape_target": top_escape,
        "state": STATE_MAP["POSITIVE"] if any_escape else STATE_MAP["NEGATIVE"]
    })

verdict_df = pd.DataFrame(verdict_rows)
verdict_df.to_csv(OUT_DIR / "exp1_3_dimension_verdicts.csv", index=False)
display(verdict_df)

,penalized_dimension,any_escape_detected,n_escape_targets,top_escape_target,state
0,length,True,2,list_density,p
1,repetition,True,3,formatting_complexity,p
2,keyword_density,True,2,list_density,p
3,hedging_density,True,1,list_density,p
4,certainty_density,True,2,formatting_complexity,p
5,list_density,False,0,None,n
6,formatting_complexity,True,2,repetition,p
7,flesch_inverse,True,2,formatting_complexity,p
8,type_token_ratio,True,1,list_density,p
9,lexical_rarity,True,1,list_density,p


In [11]:
# Cell 12 — Report
report = []
report.append("Experiment 1.3 — Per-dimension proxy penalization escape sweep")
report.append("=" * 72)
report.append(f"Pool path: {POOL_PATH}")
report.append(f"Score path: {SCORE_PATH}")
report.append(f"Dimensions tested: {SECONDARY_DIMS}")
report.append(f"Penalty weight: {PENALTY_WEIGHT}")
report.append("")
report.append("Per-dimension verdicts:")
report.append(verdict_df.to_string(index=False))
report.append("")
report.append("Interpretation guidance:")
report.append("- POSITIVE state means penalizing that dimension caused at least one other dimension")
report.append("  to rise monotonically and significantly with N, while the penalized dimension itself")
report.append("  decreased -- consistent with Type II Penalization Escape as defined in the source PDF.")
report.append("- This is a heuristic surface-metric approximation, not a manipulation of internal")
report.append("  ArmoRM MoE dimension heads, so results should be reported as proxy-level evidence only.")
report.append("")
report.append("Output files:")
for p in [CANDIDATE_METRICS_CSV, SELECTIONS_CSV, SUMMARY_CSV, ESCAPE_MATRIX_CSV,
          OUT_DIR/"exp1_3_dimension_verdicts.csv"]:
    report.append(str(p))

REPORT_TXT.write_text("\n".join(report), encoding="utf-8")
print(f"\nSaved report: {REPORT_TXT}")
print("\nDone.")


Saved report: /kaggle/working/exp1_3_per_dimension_escape/exp1_3_report.txt

Done.


In [12]:
# Cell 13 — ArmoRM raw score trajectory by penalized dimension

armorm_score_table = (
    summary.pivot(index="N", columns="condition", values="armorm_raw_mean")
    .reset_index()
)

armorm_score_table.to_csv(OUT_DIR / "exp1_3_armorm_score_trajectory.csv", index=False)
print("Saved:", OUT_DIR / "exp1_3_armorm_score_trajectory.csv")
display(armorm_score_table)

Saved: /kaggle/working/exp1_3_per_dimension_escape/exp1_3_armorm_score_trajectory.csv


condition,N,control,penalize_certainty_density,penalize_flesch_inverse,penalize_formatting_complexity,penalize_hedging_density,penalize_keyword_density,penalize_length,penalize_lexical_rarity,penalize_list_density,penalize_repetition,penalize_type_token_ratio
0,1,0.046552,0.046552,0.046552,0.046552,0.046552,0.046552,0.046552,0.046552,0.046552,0.046552,0.046552
1,4,0.079568,0.075740,0.074006,0.066537,0.067930,0.068903,0.070821,0.073434,0.064397,0.066643,0.075530
2,16,0.096433,0.094867,0.093018,0.087632,0.089579,0.090815,0.091037,0.091849,0.086701,0.087810,0.093828
3,64,0.106938,0.105701,0.104812,0.100215,0.102648,0.102859,0.104045,0.104146,0.099283,0.100289,0.105152
4,256,0.114835,0.113887,0.113079,0.108856,0.111983,0.112087,0.113258,0.112045,0.108926,0.109705,0.112737


In [13]:
# Cell 14 — ArmoRM score gain summary for each penalized dimension

gain_rows = []

for cond in summary["condition"].unique():
    block = summary[summary["condition"] == cond].sort_values("N")
    row_n1 = block.loc[block["N"] == 1]
    row_n256 = block.loc[block["N"] == 256]

    if row_n1.empty or row_n256.empty:
        continue

    armorm_n1 = float(row_n1["armorm_raw_mean"].iloc[0])
    armorm_n256 = float(row_n256["armorm_raw_mean"].iloc[0])
    selected_n1 = float(row_n1["selected_score_mean"].iloc[0])
    selected_n256 = float(row_n256["selected_score_mean"].iloc[0])

    gain_rows.append({
        "condition": cond,
        "armorm_raw_N1": armorm_n1,
        "armorm_raw_N256": armorm_n256,
        "armorm_raw_delta": armorm_n256 - armorm_n1,
        "selected_score_N1": selected_n1,
        "selected_score_N256": selected_n256,
        "selected_score_delta": selected_n256 - selected_n1,
    })

gain_df = pd.DataFrame(gain_rows).sort_values("armorm_raw_delta", ascending=False)
gain_df.to_csv(OUT_DIR / "exp1_3_armorm_gain_summary.csv", index=False)
print("Saved:", OUT_DIR / "exp1_3_armorm_gain_summary.csv")
display(gain_df)

Saved: /kaggle/working/exp1_3_per_dimension_escape/exp1_3_armorm_gain_summary.csv


,condition,armorm_raw_N1,armorm_raw_N256,armorm_raw_delta,selected_score_N1,selected_score_N256,selected_score_delta
0,control,0.046552,0.114835,0.068283,0.046552,0.114835,0.068283
1,penalize_certainty_density,0.046552,0.113887,0.067335,-0.077357,0.113887,0.191244
6,penalize_length,0.046552,0.113258,0.066707,-0.059416,0.113251,0.172667
2,penalize_flesch_inverse,0.046552,0.113079,0.066527,-0.077973,0.113026,0.190999
10,penalize_type_token_ratio,0.046552,0.112737,0.066185,-0.101525,0.112621,0.214146
5,penalize_keyword_density,0.046552,0.112087,0.065535,-0.095547,0.111998,0.207545
7,penalize_lexical_rarity,0.046552,0.112045,0.065493,-0.092848,0.111994,0.204842
4,penalize_hedging_density,0.046552,0.111983,0.065431,-0.083493,0.111859,0.195351
9,penalize_repetition,0.046552,0.109705,0.063154,-0.054587,0.109456,0.164043
8,penalize_list_density,0.046552,0.108926,0.062374,-0.069581,0.108900,0.178481


In [14]:
# Cell 15 — Spearman scaling of ArmoRM raw score vs log2(N) under each penalty

logN = [math.log2(n) for n in N_LIST]
scale_rows = []

for cond in summary["condition"].unique():
    block = summary[summary["condition"] == cond].sort_values("N")
    if len(block) != len(N_LIST):
        continue

    vals = block["armorm_raw_mean"].tolist()
    rho, p = spearmanr(logN, vals)

    scale_rows.append({
        "condition": cond,
        "armorm_raw_N1": vals[0],
        "armorm_raw_N256": vals[-1],
        "armorm_raw_delta": vals[-1] - vals[0],
        "spearman_rho": rho,
        "p_value": p,
        "armorm_still_scales": bool((vals[-1] > vals[0]) and (rho >= MIN_RHO) and (p < ALPHA))
    })

scale_df = pd.DataFrame(scale_rows).sort_values(["armorm_still_scales", "armorm_raw_delta"], ascending=[False, False])
scale_df.to_csv(OUT_DIR / "exp1_3_armorm_scaling_checks.csv", index=False)
print("Saved:", OUT_DIR / "exp1_3_armorm_scaling_checks.csv")
display(scale_df)

Saved: /kaggle/working/exp1_3_per_dimension_escape/exp1_3_armorm_scaling_checks.csv


,condition,armorm_raw_N1,armorm_raw_N256,armorm_raw_delta,spearman_rho,p_value,armorm_still_scales
0,control,0.046552,0.114835,0.068283,1.0,1.404265e-24,True
1,penalize_certainty_density,0.046552,0.113887,0.067335,1.0,1.404265e-24,True
6,penalize_length,0.046552,0.113258,0.066707,1.0,1.404265e-24,True
2,penalize_flesch_inverse,0.046552,0.113079,0.066527,1.0,1.404265e-24,True
10,penalize_type_token_ratio,0.046552,0.112737,0.066185,1.0,1.404265e-24,True
5,penalize_keyword_density,0.046552,0.112087,0.065535,1.0,1.404265e-24,True
7,penalize_lexical_rarity,0.046552,0.112045,0.065493,1.0,1.404265e-24,True
4,penalize_hedging_density,0.046552,0.111983,0.065431,1.0,1.404265e-24,True
9,penalize_repetition,0.046552,0.109705,0.063154,1.0,1.404265e-24,True
8,penalize_list_density,0.046552,0.108926,0.062374,1.0,1.404265e-24,True


In [15]:
# Cell 16 — Compare each penalized condition to control at N=256 using ArmoRM raw score

control_256 = sel_df[(sel_df["condition"] == "control") & (sel_df["N"] == 256)].copy()

compare_rows = []

for pen_dim in SECONDARY_DIMS:
    cond = f"penalize_{pen_dim}"
    test_256 = sel_df[(sel_df["condition"] == cond) & (sel_df["N"] == 256)].copy()

    if control_256.empty or test_256.empty:
        continue

    compare_rows.append({
        "condition": cond,
        "control_mean_armorm_raw": float(control_256["armorm_raw"].mean()),
        "penalized_mean_armorm_raw": float(test_256["armorm_raw"].mean()),
        "delta_pen_minus_control": float(test_256["armorm_raw"].mean() - control_256["armorm_raw"].mean()),
        "control_std_armorm_raw": float(control_256["armorm_raw"].std()),
        "penalized_std_armorm_raw": float(test_256["armorm_raw"].std()),
    })

armorm_compare_df = pd.DataFrame(compare_rows).sort_values("delta_pen_minus_control", ascending=False)
armorm_compare_df.to_csv(OUT_DIR / "exp1_3_armorm_vs_control_n256.csv", index=False)
print("Saved:", OUT_DIR / "exp1_3_armorm_vs_control_n256.csv")
display(armorm_compare_df)

Saved: /kaggle/working/exp1_3_per_dimension_escape/exp1_3_armorm_vs_control_n256.csv


,condition,control_mean_armorm_raw,penalized_mean_armorm_raw,delta_pen_minus_control,control_std_armorm_raw,penalized_std_armorm_raw
4,penalize_certainty_density,0.114835,0.113887,-0.000948,0.027226,0.027382
0,penalize_length,0.114835,0.113258,-0.001576,0.027226,0.027632
7,penalize_flesch_inverse,0.114835,0.113079,-0.001756,0.027226,0.027189
8,penalize_type_token_ratio,0.114835,0.112737,-0.002098,0.027226,0.026997
2,penalize_keyword_density,0.114835,0.112087,-0.002748,0.027226,0.027757
9,penalize_lexical_rarity,0.114835,0.112045,-0.002790,0.027226,0.027114
3,penalize_hedging_density,0.114835,0.111983,-0.002851,0.027226,0.028212
1,penalize_repetition,0.114835,0.109705,-0.005129,0.027226,0.028993
5,penalize_list_density,0.114835,0.108926,-0.005909,0.027226,0.027370
6,penalize_formatting_complexity,0.114835,0.108856,-0.005979,0.027226,0.027742


In [16]:
# Cell 17 — Merge per-dimension escape verdicts with ArmoRM score behavior

verdict_enriched = verdict_df.copy()
verdict_enriched["condition"] = verdict_enriched["penalized_dimension"].apply(lambda d: f"penalize_{d}")

verdict_enriched = verdict_enriched.merge(
    scale_df[["condition", "armorm_raw_delta", "spearman_rho", "p_value", "armorm_still_scales"]],
    on="condition",
    how="left"
)

verdict_enriched = verdict_enriched.merge(
    gain_df[["condition", "selected_score_delta"]],
    on="condition",
    how="left"
)

verdict_enriched.to_csv(OUT_DIR / "exp1_3_verdicts_with_armorm_cache.csv", index=False)
print("Saved:", OUT_DIR / "exp1_3_verdicts_with_armorm_cache.csv")
display(verdict_enriched)

Saved: /kaggle/working/exp1_3_per_dimension_escape/exp1_3_verdicts_with_armorm_cache.csv


,penalized_dimension,any_escape_detected,n_escape_targets,top_escape_target,state,condition,armorm_raw_delta,spearman_rho,p_value,armorm_still_scales,selected_score_delta
0,length,True,2,list_density,p,penalize_length,0.066707,1.0,1.404265e-24,True,0.172667
1,repetition,True,3,formatting_complexity,p,penalize_repetition,0.063154,1.0,1.404265e-24,True,0.164043
2,keyword_density,True,2,list_density,p,penalize_keyword_density,0.065535,1.0,1.404265e-24,True,0.207545
3,hedging_density,True,1,list_density,p,penalize_hedging_density,0.065431,1.0,1.404265e-24,True,0.195351
4,certainty_density,True,2,formatting_complexity,p,penalize_certainty_density,0.067335,1.0,1.404265e-24,True,0.191244
5,list_density,False,0,None,n,penalize_list_density,0.062374,1.0,1.404265e-24,True,0.178481
6,formatting_complexity,True,2,repetition,p,penalize_formatting_complexity,0.062304,1.0,1.404265e-24,True,0.164573
7,flesch_inverse,True,2,formatting_complexity,p,penalize_flesch_inverse,0.066527,1.0,1.404265e-24,True,0.190999
8,type_token_ratio,True,1,list_density,p,penalize_type_token_ratio,0.066185,1.0,1.404265e-24,True,0.214146
9,lexical_rarity,True,1,list_density,p,penalize_lexical_rarity,0.065493,1.0,1.404265e-24,True,0.204842


In [17]:
# Cell 18 — Auto-interpretation from cached ArmoRM behavior

n_escape = int(verdict_df["any_escape_detected"].sum())
n_total = int(len(verdict_df))

n_escape_and_scale = int(
    verdict_enriched[
        (verdict_enriched["any_escape_detected"] == True) &
        (verdict_enriched["armorm_still_scales"] == True)
    ].shape[0]
)

best_escape_row = verdict_enriched.sort_values(
    ["any_escape_detected", "armorm_raw_delta", "selected_score_delta"],
    ascending=[False, False, False]
).iloc[0]

print("ArmoRM cache interpretation")
print("=" * 60)
print(f"Dimensions with proxy escape detected: {n_escape}/{n_total}")
print(f"Escape + continued ArmoRM scaling: {n_escape_and_scale}/{n_total}")
print(f"Best-supported penalized dimension: {best_escape_row['penalized_dimension']}")
print(f"Top escape target: {best_escape_row['top_escape_target']}")
print(f"ArmoRM raw delta (N256 - N1): {best_escape_row['armorm_raw_delta']:.6f}")
print(f"Selected score delta (N256 - N1): {best_escape_row['selected_score_delta']:.6f}")
print(f"ArmoRM scaling rho: {best_escape_row['spearman_rho']:.6f}")
print(f"ArmoRM scaling p-value: {best_escape_row['p_value']:.6g}")

ArmoRM cache interpretation
Dimensions with proxy escape detected: 9/10
Escape + continued ArmoRM scaling: 9/10
Best-supported penalized dimension: certainty_density
Top escape target: formatting_complexity
ArmoRM raw delta (N256 - N1): 0.067335
Selected score delta (N256 - N1): 0.191244
ArmoRM scaling rho: 1.000000
ArmoRM scaling p-value: 1.40427e-24
